In [16]:
import sys
import time

In [17]:
sys.path.append("../..")

In [18]:
from matplotlib import pyplot as plt 
from tqdm import tqdm

import numpy as np
import pandas as pd

import torch
from torchvision import transforms
from torch import nn
from torch.nn import functional as F

from xaikd import models, datasets, utils, attributors
from xaikd.utils import metrics

from zennit.torchvision import ResNetCanonizer
from zennit.composites import EpsilonGammaBox
from zennit.attribution import Gradient

from torch.utils.data import random_split

from scipy.stats import ortho_group
from datetime import datetime

import timm

In [19]:
device = utils.get_device()
device

'cuda'

In [20]:
dataset = datasets.construct("imagenet-butterfly")

In [21]:
MODEL_NAME = "dm_nfnet_f0"

In [22]:
# teacher = models.get_trained_model(MODEL_NAME)
# utils.modify_last_layer_for_subclasses(teacher, dataset.selected_classes)


teacher = timm.create_model(MODEL_NAME, pretrained=True).eval()
setattr(teacher, "__last_layer", teacher.head.fc)

utils.modify_last_layer_for_subclasses(teacher, dataset.selected_classes)
teacher.to(device)

NormFreeNet(
  (stem): Sequential(
    (conv1): ScaledStdConv2dSame(3, 16, kernel_size=(3, 3), stride=(2, 2))
    (act2): GammaAct()
    (conv2): ScaledStdConv2dSame(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (act3): GammaAct()
    (conv3): ScaledStdConv2dSame(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (act4): GammaAct()
    (conv4): ScaledStdConv2dSame(64, 128, kernel_size=(3, 3), stride=(2, 2))
  )
  (stages): Sequential(
    (0): Sequential(
      (0): NormFreeBlock(
        (downsample): DownsampleAvg(
          (pool): Identity()
          (conv): ScaledStdConv2dSame(128, 256, kernel_size=(1, 1), stride=(1, 1))
        )
        (act1): GammaAct()
        (conv1): ScaledStdConv2dSame(128, 128, kernel_size=(1, 1), stride=(1, 1))
        (act2): GammaAct()
        (conv2): ScaledStdConv2dSame(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act2b): GammaAct()
        (conv2b): ScaledStdConv2dSame(128, 128, kernel_size=(3, 

In [23]:
teacher.head.fc.weight.shape

torch.Size([6, 3072])

In [24]:
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from torchvision import transforms as T

def get_input_transform(model):
  # timm provides a way to resolve (or get) transformation used by a particular model
  # via `resolve_data_config(...)`
  config = resolve_data_config({}, model=model)
  transform = create_transform(**resolve_data_config({}, model=model))

  return transform

input_transform = get_input_transform(teacher)
input_transform

Compose(
    Resize(size=(213, 213), interpolation=bicubic, max_size=None, antialias=warn)
    CenterCrop(size=(192, 192))
    ToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

In [10]:
rc_transform = T.Compose(input_transform.transforms[:2])

In [11]:
dataset.input_transformation

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [12]:
dataset.input_transformation = input_transform
dataset.input_transformation

Compose(
    Resize(size=(278, 278), interpolation=bicubic, max_size=None, antialias=warn)
    CenterCrop(size=(256, 256))
    ToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

In [13]:
ds_train = dataset.create_subset(train_split=True)
# ds_train, _ = random_split(ds_train, [0.1, 0.9])
ds_val = dataset.create_subset(train_split=False)

We have 7800 images in classes [321, 322, 323, 324, 325, 326]


preparing `ImageNetButterfly` samples: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 7800/7800 [00:00<00:00, 829460.25it/s]


We have 300 images in classes [321, 322, 323, 324, 325, 326]


preparing `ImageNetButterfly` samples: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:00<00:00, 907858.01it/s]


In [14]:
MAIN_LAYER = "stages.1"

In [15]:
teacher_acc, _ = metrics.accuracy(
    teacher,
    dataloader=datasets.build_dataloader(ds_val, shuffle=False),
    num_classes=dataset.num_classes,
    device=device,
)
teacher_acc

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB (GPU 0; 39.39 GiB total capacity; 38.50 GiB already allocated; 1.94 MiB free; 38.83 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

# Implementation Attribution

In [ ]:
from matplotlib.colors import ListedColormap


from PIL import Image
import requests

from zennit.core import Composite
from zennit.attribution import Gradient

## Helper Function

In [ ]:
image_collection = (
    ("volcano.jpg", 980),
    ("zebra.jpg", 340),
    ("castle.jpg", 483),
    ("castle.jpg", 919),
    ("viaduct.jpg", 888),
)

def get_image(ix):
  name, target_label = image_collection[ix]

#   url = f"https://pchormai-public.s3.amazonaws.com/imagenet-samples/dev-images/{name}"
  url = f"https://github.com/p16i/drsa-demo/blob/main/tests/data/{name}?raw=true"
  img = Image.open(requests.get(url, stream=True).raw)

  return img, target_label

def ano():

    img, target_label = get_image(4)

    return img

ano()

In [ ]:
# This function will be used to visualize the heatmap.
# It is adapted from https://git.tu-berlin.de/gmontavon/lrp-tutorial/-/blob/main/utils.py.
def plot_heatmap(heatmap):
    """
    args:
        heatmap np.array(h, w):
        reference_heatmap (np.array(h, w), optional): used for calculating normalization values. Defaults to None.
        total_score (float, optional): used for normalizing scores. Defaults to None.
    """


    b = 10 * ((np.abs(heatmap) ** 3.0).mean() ** (1.0 / 3))

    my_cmap = plt.cm.seismic(np.arange(plt.cm.seismic.N))
    my_cmap[:, 0:3] *= 0.85
    my_cmap = ListedColormap(my_cmap)

    sum_Ri = np.sum(heatmap)

    txt = r"$\sum R =%.2f$" % (sum_Ri)

    # plt.imshow(heatmap, cmap="bwr", vmin=-b, vmax=b)
    plt.imshow(heatmap, cmap=my_cmap, vmin=-b, vmax=b)

    h = heatmap.shape[0]
    plt.xlabel(txt)

plot_heatmap(np.random.randn(28, 28))


## Core Functions

In [ ]:
from zennit.core import  Composite, BasicHook, Stabilizer, expand, ParamMod, Hook

from zennit.rules import ZBox, Pass, Epsilon, Gamma, Flat, GammaMod, NoMod, zero_bias
from zennit.types import Linear


from zennit.attribution import Gradient
from zennit.canonizers import AttributeCanonizer
from functools import partial


from timm.models.layers import SelectAdaptivePool2d, SEModule, pad_same
from timm.models.nfnet import GammaAct, ScaledStdConv2dSame, NormFreeBlock, DownsampleAvg


LB = (0-input_transform.transforms[-1].mean) /  input_transform.transforms[-1].std
HB = (1-input_transform.transforms[-1].mean) /  input_transform.transforms[-1].std


In [ ]:

class ConstantMul(torch.nn.Module):
    def __init__(self, constant):
        super().__init__()
        self.constant = constant

    def forward(self, x):
        return x * self.constant


class Summation(torch.nn.Module):
    def forward(self, x):
        return x.sum(dim=-1)


class GammaForPooling(BasicHook):
    def __init__(self, gamma=0.25, stabilizer=1e-6, zero_params=None):
        mod_kwargs = {'zero_params': zero_params}
        mod_kwargs_nobias = {'zero_params': zero_bias(zero_params)}
        stabilizer_fn = Stabilizer.ensure(stabilizer)
        
        super().__init__(
            input_modifiers=[
                lambda input: input.clamp(min=0) * (1+gamma),
                lambda input: input.clamp(max=0),
                lambda input: input.clamp(min=0) ,
                lambda input: input.clamp(max=0) * (1+gamma),
                lambda input: input,
            ],
            param_modifiers=[
                NoMod(),
                NoMod(),
                NoMod(),
                NoMod(),
                NoMod(),
            ],
            output_modifiers=[lambda output: output] * 5,
            gradient_mapper=(
                lambda out_grad, outputs: [
                    output * out_grad / stabilizer_fn(denom)
                    for output, denom in (
                        [(outputs[4] > 0., sum(outputs[:2]))] * 2
                        + [(outputs[4] < 0., sum(outputs[2:4]))] * 2
                    )
                ] + [torch.zeros_like(out_grad)]
            ),
            reducer=(
                lambda inputs, gradients: sum(input * gradient for input, gradient in zip(inputs[:4], gradients[:4]))
            ),
        )

class PassWithConstantSign(Hook):
    '''Unmodified pass-through rule.
    If the rule of a layer shall not be any other, is elementwise and shall not be the gradient, the `Pass` rule simply
    passes upper layer relevance through to the lower layer.
    '''
    def backward(self, module, grad_input, grad_output):
        '''Pass through the upper gradient, skipping the one for this layer.'''

        if isinstance(module.constant, torch.Tensor):
            sign = torch.sign(module.constant.detach())
        else:
            sign = np.sign(module.constant)

        sign = float(sign)

        return tuple([sign * grad for grad in grad_output])


class ScaledStdConv2dSameCanonizer(AttributeCanonizer):
    '''Canonizer specifically for Bottlenecks of torchvision.models.resnet* type models.'''
    def __init__(self):
        super().__init__(self._attribute_map)

    @classmethod
    def _attribute_map(cls, name, module):
        if isinstance(module, ScaledStdConv2dSame):

            std, mean = torch.std_mean(
                module.weight, dim=[1, 2, 3], keepdim=True, unbiased=False
            )
            weight = module.scale * (module.weight - mean) / (std + module.eps)

            attributes = {
                'forward': cls.forward.__get__(module),
                'weight': nn.Parameter(weight),
                '___original_weight': module.weight
            }
            return attributes

        return None

    @staticmethod
    def forward(self, x):
        if self.same_pad:
            x = pad_same(x, self.kernel_size, self.stride, self.dilation)

        weight = self.weight
        return F.conv2d(x, self.gain * weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

    def remove(self):
        original_weight = getattr(self.module, "___original_weight")
        
        # remove all the keys
        for key in self.attribute_keys:
            delattr(self.module, key)

        # rollback to the original weight
        setattr(self.module, "weight", original_weight)

    def copy(self):
        # pat: we have to do this, otherwise copy() comes from AttributeCanonizer
        return ScaledStdConv2dSameCanonizer()


class SEModuleCanonizer(AttributeCanonizer):
    '''Canonizer specifically for Bottlenecks of torchvision.models.resnet* type models.'''
    def __init__(self):
        super().__init__(self._attribute_map)

    @classmethod
    def _attribute_map(cls, name, module):
        if isinstance(module, SEModule):
            attributes = {
                'forward': cls.forward.__get__(module),
            }
            return attributes
        return None

    @staticmethod
    def forward(self, x):
        x_se = x.detach().mean((2, 3), keepdim=True)

        x_se = self.fc1(x_se)
        x_se = self.act(x_se)
        x_se = self.fc2(x_se)

        gate = self.gate(x_se)

        out = x * gate

        return out


class NormFreeBlockCanonizer(AttributeCanonizer):
    def __init__(self):
        super().__init__(self._attribute_map)

    @classmethod
    def _attribute_map(cls, name, module):
        if isinstance(module, NormFreeBlock):
            attributes = {
                'forward': cls.forward.__get__(module),
                'shortcut_summation': Summation(),
                'mul_alpha': ConstantMul(module.alpha),
                'mul_beta': ConstantMul(module.beta),
                'mul_attn_gain': ConstantMul(module.attn_gain),
                'mul_skipinit_gain': ConstantMul(module.skipinit_gain),
            }
            return attributes

        return None

    @staticmethod
    def forward(self, x):
        shortcut = x * 1.0

        out_act1 = self.mul_beta(self.act1(x))


        residue_inp = out_act1 * 1.0


        # residual branch
        out = self.conv1(residue_inp)
        out = self.conv2(self.act2(out))
        if self.conv2b is not None:
            out = self.conv2b(self.act2b(out))
        if self.attn is not None:
            raise
            out = self.attn_gain * self.attn(out)

        out_conv3 = self.conv3(self.act3(out))

        if self.attn_last is not None:
            out_attn_gain = self.mul_attn_gain(self.attn_last(out_conv3))


        out = self.drop_path(out_attn_gain)

        if self.skipinit_gain is not None:
            # @pat: why does this value matter? what is the inuition the authors of NFNets?
            out = self.mul_skipinit_gain(out)

        residue_output = self.mul_alpha(out)

        if self.downsample is not None :
            inp_shortcut = out_act1 * 1
            shortcut = self.downsample(inp_shortcut)


        concat = torch.stack([residue_output, shortcut], dim=-1)
        # this is the only line that is different from the original function.
        out = self.shortcut_summation(concat)

        return out

def module_map(ctx, name, module, gamma, eps):
    try:
        next(module.children())
    except StopIteration:
        # StopIteration is raised if the iterator has no more elements,
        # which means in this case there are no children and module is a leaf
        pass
    else:
        # pat: find a way to handle this case outside try
        if isinstance(module, SEModule):
            return Pass()

        # if StopIteration is not raised on the first element, module is not a leaf
        return None

    # count the number of the leaves processed yet in 'leafnum'
    if 'leafnum' not in ctx:
        ctx['leafnum'] = 0
    else:
        ctx['leafnum'] += 1


    leafnum = ctx["leafnum"]

    if leafnum == 0:
        assert isinstance(module, ScaledStdConv2dSame)
        return ZBox(
            low=LB.reshape(1, -1, 1, 1), 
            high=HB.reshape(1, -1, 1, 1)
        )
    elif isinstance(module, (ScaledStdConv2dSame, nn.Linear)): 
        return Gamma(gamma=gamma, stabilizer=eps)
    elif isinstance(module, (Summation, nn.AvgPool2d, nn.AdaptiveAvgPool2d)):
        return GammaForPooling(gamma=gamma, stabilizer=eps)
    elif isinstance(module, (GammaAct, ConstantMul)):
        return Pass()
    else:
        return None

In [ ]:
def get_attributor(model, gamma=0.1, eps=1e-3):

    composite = Composite(
            module_map=partial(module_map, gamma=gamma, eps=eps), 
            canonizers=[
                NormFreeBlockCanonizer(),
                SEModuleCanonizer(),
                ScaledStdConv2dSameCanonizer()
            ]
        )

    return Gradient(model, composite)


def explain(img, label):


    arr_gammas = [
        # 0, 0.001,
        0.01, 0.1, 1]
    eps = 1e-3
    
    ncols = len(arr_gammas) + 1
    plt.figure(figsize=(2*ncols, 2))

    model = timm.create_model("dm_nfnet_f0", pretrained=True).eval()
    model.to(device)

    for gix, gamma in tqdm(enumerate(arr_gammas), total=len(arr_gammas)):
        with get_attributor(
            model, 
            gamma=gamma, eps=eps
        ) as attributor:  
            xp = input_transform(img).unsqueeze(0)
            
            out, hm = attributor(
                xp.to(device),
                lambda logits: logits * F.one_hot(torch.tensor([label]), num_classes=1000).to(logits.device)
            )
            out = out.detach().cpu()
            hm = hm.detach().cpu()

            if gix == 0:
                fx = out[0, label]
            
                plt.subplot(1, ncols, 1)
                plt.imshow(rc_transform(img))
                plt.xlabel(f"$f({label})={fx:.2f}$")
                plt.ylabel(f"Target Label: {label}")
                plt.xticks([]); plt.yticks([])

            plt.subplot(1, ncols, 2 + gix )
            hm = hm.squeeze().sum(axis=0).detach().numpy()
            plot_heatmap(hm)
            plt.title(f"$\gamma={gamma:.2f}$")
            plt.xticks([]); plt.yticks([])

    plt.show()

explain(*get_image(2))

In [ ]:
for i in range(len(image_collection)):
    explain(*get_image(i))

# Collect Activation and Context Vectors

In [ ]:

    
def extract_activation_context(
    model,
    layer,
    dataset,
    data_loader,
    logit_modifier,
    rng,
    device="cpu",
    number_of_selected_spatial_locations=20,
    verbose=False
):
    arr_act = []
    arr_ctx = []

    
    with get_attributor(model) as attributor:
        for batch in tqdm(data_loader, desc=f"layer={layer}"):
            x, y = batch
            x = x.to(device)

            try:
                module, hook = utils.interceptor.attach_hook_intercept_layer_output(
                    model, layer, should_retain_grad=True
                )

                _ = attributor.forward(x, lambda logits: logit_modifier(logits, y))

                act = utils.interceptor.get_output(module)
                rel = act.grad
                

                output_dimensions = act.shape[1:]
                # print("output.dimension", output_dimensions)
                # todo: check this with Gregoire again!
                ctx = torch.where(
                    act.abs() > 0, 
                    rel / act, 
                    0
                )

                np.testing.assert_allclose(
                    (act * ctx).detach().cpu().numpy(), 
                    (rel * (act.abs() > 0)).detach().cpu().numpy(),
                    atol=1e-6
                )

                assert ctx.shape == act.shape

                act = act.detach().cpu().numpy()
                ctx = ctx.detach().cpu().numpy()

                if len(act.shape) == 2:
                    act = act[:, :, None, None]
                    ctx = ctx[:, :, None, None]

                selected_act, selected_ctx = utils.subsample_tensors(
                    act,
                    ctx,
                    num_locations=number_of_selected_spatial_locations,
                    rng=rng,
                )
                arr_act.append(selected_act)
                arr_ctx.append(selected_ctx)

            finally:
                hook.remove()
    if verbose:
        print(f"{layer}: output-dims={output_dimensions}")

    arr_act = np.vstack(arr_act)
    arr_ctx = np.vstack(arr_ctx)

    return arr_act, arr_ctx



def get_act_ctx(model, dataset, layer, verbose=False):

    logit_modifier = attributors.WinningClassOneHotEvidence(num_classes=len(dataset.selected_classes))

    rng = np.random.default_rng(seed=1)

    dl = datasets.build_dataloader(
        ds_train,
        shuffle=False
    )
    return extract_activation_context(
        model=model,
        layer=layer,
        dataset=dataset,
        data_loader=dl,
        logit_modifier=logit_modifier,
        rng=rng,
        device=device,

        verbose=verbose,
    )


_arr_act, _arr_ctx = get_act_ctx(
    teacher,
    dataset,
    layer=MAIN_LAYER, 
    verbose=False
)

In [ ]:
_arr_act.shape

In [ ]:
class BasisTransform:
    def rank_k_encoder(self, k: int):
        raise
    
    def rank_k_decoder(self, k: int):
        raise
        
    def get_hook_rank_k_transformation(self, k, device):
        
        mat_enc = self.rank_k_encoder(k)
        mat_dec = self.rank_k_decoder(k)

        # X @ U @ U.T
        mat = torch.from_numpy(mat_enc @ mat_dec)

        mat = mat.unsqueeze(2).unsqueeze(3).to(device)
        mean = torch.from_numpy(self.mean).reshape(1, -1, 1, 1).to(device)
        
        def hook_func(module, inp, out):
            return F.conv2d(
                out - mean, 
                mat
            ) + mean

        return hook_func
    
class PCA(BasisTransform):
    def __init__(self, model, layer, arr_act, arr_ctx, centering=False):
        self.mean =  arr_act.mean(axis=0)
        self.centering = centering
        
        if not centering:
            self.mean = self.mean*0
            
        eigvals, eigvecs = np.linalg.eigh(
            (arr_act - self.mean).T @ (arr_act - self.mean)
        )
        
        # descending sort
        sorted_indices = np.argsort(-eigvals)
        self.eigvals = eigvals[sorted_indices]
        self.eigvecs = eigvecs[:, sorted_indices]
    
    def rank_k_encoder(self, k: int):
        # X @ U
        return self.eigvecs[:, :k]
    
    def rank_k_decoder(self, k: int):
        # Z @ U.T
        return self.eigvecs[:, :k].T


    def __str__(self):
        return f"PCA(centering={self.centering})"

print(PCA(teacher, MAIN_LAYER, _arr_act, _arr_ctx))

In [ ]:
class PRCASortAbs(BasisTransform):
    def __init__(self, model, layer, arr_act, arr_ctx, centering=False):
        self.mean =  arr_act.mean(axis=0)

        self.centering = centering
        if not self.centering:
            self.mean = self.mean * 0

        ccov = (arr_act - self.mean).T @ arr_ctx  + arr_ctx.T @ (arr_act - self.mean)
        
        eigvals, eigvecs = np.linalg.eigh(ccov)
        
        # descending sort
        sorted_indices = np.argsort(-np.abs(eigvals))
        self.eigvals = eigvals[sorted_indices]
        self.eigvecs = eigvecs[:, sorted_indices]
    
    def rank_k_encoder(self, k: int):
        # X @ U
        return self.eigvecs[:, :k]
    
    def rank_k_decoder(self, k: int):
        # Z @ U.T
        return self.eigvecs[:, :k].T


    def __str__(self):
        return f"PRCASortAbs(centering={self.centering})"
print(PRCASortAbs(teacher, MAIN_LAYER, _arr_act, _arr_ctx))

In [ ]:
def compute_accuracy_with_Uk(
    model, layer, Uk, dataset, dl
):
    module = utils.interceptor.get_module(model, layer)

    UUT, hook_func = fh_low_rank(Uk)
    hook = module.register_forward_hook(hook_func)

    try:
        acc, _ = metrics.accuracy(
                        model,
                        dataloader=dl,
                        num_classes=dataset.num_classes,
                        device=device,
                )
        
    finally:
        hook.remove()
        del UUT
    return acc  


def fh_low_rank(U):
    d, k = U.shape

    assert k <= d

    UUT = U @ U.T 
    assert UUT.shape == (d, d)

    UUT = torch.from_numpy(UUT).float().unsqueeze(2).unsqueeze(3).to(device)
    
    def do(module, inp, out):
        inp, = inp
        
        out2 = F.conv2d(out, UUT)

        if k == d and False:
            np.testing.assert_allclose(
                out.detach().cpu().numpy(), out2.detach().cpu().numpy(),
                atol=1e-3
            )

        
        return out2

    return UUT, do 

In [ ]:
def compute_accuracy_with_basis(
    model, layer, basis, k, dataset, dataloader
):
    module = utils.interceptor.get_module(model, layer)

    hook_func = basis.get_hook_rank_k_transformation(k, device=device)
    hook = module.register_forward_hook(hook_func)

    try:
        acc, _ = metrics.accuracy(
            model,
            dataloader=dataloader,
            num_classes=dataset.num_classes,
            device=device,
        )
        
    finally:
        hook.remove()

    return acc  

In [ ]:
import matplotlib as mpl


def compute_accuracy_of_basis_at_k(
    model, dataset, layer, 
    arr_ks,
    arr_bases,
):
    dl_val = datasets.build_dataloader(ds_val, shuffle=False)
    
    rows = []
    print(arr_bases)
    for k in tqdm(arr_ks):
        for basis in arr_bases:
    
            acc_val = compute_accuracy_with_basis(
                model=model, 
                layer=layer, 
                basis=basis,
                k=k,
                dataset=dataset, 
                dataloader=dl_val
            )

            rows.append(
                dict(
                    layer=layer, k=k, 
                    basis_name=f"{basis}",
                    acc_val=acc_val,
                )
            )
    

    df = pd.DataFrame(rows)

    return df


def estimate_rank_k_accuracies(
    model, dataset, layer,
    arr_bases,
    arr_ks=[1,2, 4, 8, 16, 32, 48, 64],
):

    colors = mpl.cm.Blues
    nbases = len(arr_bases)

    df = compute_accuracy_of_basis_at_k(
        model, dataset, layer, 
        arr_bases=arr_bases,
        arr_ks=arr_ks
    )

    torch.cuda.empty_cache()

    def alias(name):
        return name
        if name == "pca":
            return "PCA"
        elif "prcaopt:" in name:
            _, slug = name.split(":")
            return f"PCA-LH:{slug}"
        else:
            raise
            
    def ls(name):
        if name in  ["pca", "prca"]:
            return "--"
        else:
            return "-"

    def color(name):
        if name == "pca":
            return "blue"
        elif name == "prca":
            return "red"
        else:
            return None
            

    cols = ["acc_val"]
    ncols = len(cols)
    
  
    
    return df
    
estimate_rank_k_accuracies(
    model=teacher,
    dataset=dataset,
    layer=MAIN_LAYER,
    arr_bases=[
        PCA(teacher, MAIN_LAYER, _arr_act, _arr_ctx),
        PRCASortAbs(teacher, MAIN_LAYER, _arr_act, _arr_ctx),
        # PCALookAhead(teacher, MAIN_LAYER, _arr_act, _arr_ctx),
    ],
)

In [ ]:
def get_accuracy_layers(
    arr_layers,
    arr_basis_classes,
):

    arr_stats = []

    ncols = len(arr_layers)
    
    plt.figure(figsize=(3*ncols, 3))
    plt.suptitle(MODEL_NAME)
    
    for lix, layer in enumerate(arr_layers):
        plt.subplot(1, ncols, lix+1)
        
        plt.axhline(teacher_acc, ls="--", color="black", label=f"teacher_acc={teacher_acc:.4f}")
        plt.title(layer)

        arr_act, arr_ctx = get_act_ctx(
            teacher,
            dataset,
            layer=layer, 
            verbose=False
        )

        for basis_class in arr_basis_classes:
            basis = basis_class(teacher, layer, arr_act, arr_ctx)

            _, d = arr_act.shape
            
            arr_ks = np.power(2, np.arange(np.log2(d // 4) + 1))
    
            df = compute_accuracy_of_basis_at_k(
                teacher, dataset, layer, 
                arr_bases=[basis],
                arr_ks=arr_ks.astype(int).tolist()
            )
    
            plt.plot(df.k, df.acc_val, label=f"{basis}")
            
        if lix == 0:
            plt.ylabel("Accuracy")
        
        plt.legend()

get_accuracy_layers(
    arr_layers=[
        "stages.0", 
        "stages.1",  
        "stages.2",  
        "stages.3",  
    ],
    arr_basis_classes=[PCA, PRCASortAbs]
)

In [ ]:
def compute_cascade_accuracy(
    arr_layers,
    arr_dims,
    arr_bases,
    centering
):
    dl_val = datasets.build_dataloader(ds_val, shuffle=False)

    arr_trained_bases = dict()
    print("Training Bases")
    for layer in arr_layers:
        arr_act, arr_ctx = get_act_ctx(
            teacher,
            dataset,
            layer=layer, 
            verbose=False
        )

        for bix, basis_cls in enumerate(arr_bases):
            basis_obj = basis_cls(teacher, layer, arr_act, arr_ctx, centering=centering)
            slug = f"{basis_obj}"

            if slug not in arr_trained_bases:
                arr_trained_bases[slug] = dict()    
                
            arr_trained_bases[slug][layer] = basis_obj

    arr_stats = []
    for basis_slug in tqdm(arr_trained_bases.keys(), desc="computer acc"):
        for seq_dims in arr_dims:
            print(f"[basis_slug={basis_slug}] seq_dims={seq_dims}")

            arr_hooks = []
            try:
                for layer, k in zip(arr_layers, seq_dims):
                    basis = arr_trained_bases[basis_slug][layer]
                    module = utils.interceptor.get_module(teacher, layer)
                    hook_func = basis.get_hook_rank_k_transformation(k, device=device)
                    arr_hooks.append(module.register_forward_hook(hook_func))
                    
                val_acc, _ = metrics.accuracy(
                    teacher,
                    dataloader=dl_val,
                    num_classes=dataset.num_classes,
                    device=device,
                )
            finally:
                for hook in arr_hooks:
                    hook.remove()

            arr_stats.append(dict(
                basis=basis_slug,
                dims=",".join(np.array(seq_dims).astype(str)),
                val_acc=val_acc
            ))

    return pd.DataFrame(arr_stats)

df_cascade = compute_cascade_accuracy(
    arr_layers=["stages.0",  "stages.1", "stages.2",  "stages.3"],
    arr_dims=[
        (64, 56, 48, 40),

        (48, 40, 32, 24),

        (40, 32, 24, 16),

        (32, 24, 16, 8),
        
        (24, 16, 8, 4),
    ],
    arr_bases=[
        PCA, 
        PRCASortAbs
    ],
    centering=False
)

In [ ]:
df_cascade

In [ ]:
def viz_cascade(df):

    arr_dims = df.dims.unique()
    nsteps = len(arr_dims)

    ticks = np.arange(nsteps) * 4


    plt.figure(figsize=(5, 4))

    plt.title(MODEL_NAME)
    plt.axvline(teacher_acc, ls="--", color="black", label=f"teacher_acc={teacher_acc:.4f}")
    plt.xlabel("Accuracy")
    for bix, basis in enumerate(df.basis.unique()):
        _df = df[df.basis == basis]
        plt.barh(
            ticks + bix, 
            _df["val_acc"],
            label=basis
        )
    plt.yticks(ticks + 0.5 , arr_dims)
    plt.legend(loc="right",bbox_to_anchor=(1.55, 0.5))
viz_cascade(df_cascade)

In [ ]:
viz_cascade(
    compute_cascade_accuracy(
    arr_layers=["stages.0",  "stages.1", "stages.2",  "stages.3"],
    arr_dims=[
        (64, 56, 48, 40),

        (48, 40, 32, 24),

        (40, 32, 24, 16),

        (32, 24, 16, 8),
        
        (24, 16, 8, 4),
    ],
    arr_bases=[
        PCA, 
        PRCASortAbs
    ],
    centering=True
)
)

In [ ]:
print(f"Finished at {datetime.now()}")